In [51]:
from __future__ import annotations

import datetime as dt
import xml.etree.ElementTree as ET
from typing import Any, Dict, List, Optional
from zoneinfo import ZoneInfo

import requests

In [52]:
BASE_URL = "https://fr24api.flightradar24.com/api"
TIMEOUT = 30

In [53]:
def iso_z(dt_obj: dt.datetime) -> str:
    return (
        dt_obj.astimezone(dt.timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def parse_datetime_maybe(value: Any) -> Optional[dt.datetime]:
    if value is None:
        return None

    if isinstance(value, (int, float)):
        try:
            return dt.datetime.fromtimestamp(value, tz=dt.timezone.utc)
        except Exception:
            return None

    if not isinstance(value, str):
        return None

    s = value.strip()
    if not s:
        return None

    if s.isdigit():
        try:
            return dt.datetime.fromtimestamp(int(s), tz=dt.timezone.utc)
        except Exception:
            pass

    if s.endswith("Z"):
        s = s[:-1] + "+00:00"

    try:
        parsed = dt.datetime.fromisoformat(s)
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=dt.timezone.utc)
        return parsed.astimezone(dt.timezone.utc)
    except Exception:
        return None


def ensure_list(payload: Any) -> List[Any]:
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("data", "results", "items", "flights"):
            val = payload.get(key)
            if isinstance(val, list):
                return val
    return []

In [177]:
class FR24Client:
    def __init__(self, token: str, verbose: bool = False):
        if not token:
            raise ValueError("Missing FR24 token.")
        self.verbose = verbose
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
            "Accept-Version": "v1",
            "User-Agent": "fr24-notebook-export/1.0",
        })
        self._airport_timezone_cache: Dict[str, Optional[str]] = {}

    def get(self, path: str, params) -> Any:
        url = f"{BASE_URL}{path}"
        if self.verbose:
            print(f"GET {url}")
            print(f"params={params}")
        resp = self.session.get(url, params=params, timeout=TIMEOUT)
        if self.verbose:
            print("status =", resp.status_code)
            try:
                print(resp.text[:2000])
            except Exception:
                pass
        resp.raise_for_status()
        return resp.json()

    def flight_summary_full(
        self,
        flight_number: str,
        start_utc: dt.datetime,
        end_utc: dt.datetime,
        limit: int = 50,
    ) -> List[Dict[str, Any]]:
        params = [
            ("flights", flight_number),
            ("flight_datetime_from", iso_z(start_utc)),
            ("flight_datetime_to", iso_z(end_utc)),
            ("limit", str(limit)),
        ]
        payload = self.get("/flight-summary/full", params=params)
        return [x for x in ensure_list(payload) if isinstance(x, dict)]

    def airport_full(self, code: str) -> Dict[str, Any]:
        return self.get(f"/static/airports/{code}/full", params={})

    def get_airport_timezone(self, code: str) -> Optional[str]:
        if not code:
            return None
    
        if code in self._airport_timezone_cache:
            return self._airport_timezone_cache[code]
    
        airport = self.airport_full(code)
    
        tz_name = None
        timezone_obj = airport.get("timezone")
    
        if isinstance(timezone_obj, dict):
            tz_name = (
                timezone_obj.get("name")
                or timezone_obj.get("olson")
                or timezone_obj.get("tz")
                or timezone_obj.get("zone")
            )
        elif isinstance(timezone_obj, str):
            tz_name = timezone_obj
    
        self._airport_timezone_cache[code] = tz_name
        return tz_name

    def flight_tracks(self, fr24_id: str) -> List[Dict[str, Any]]:
        # On your account this worked with "flight_id"
        params = {
            "flight_id": fr24_id,
        }
        payload = self.get("/flight-tracks", params=params)
        if not isinstance(payload, list):
            raise RuntimeError("Unexpected flight-tracks response shape.")
        return payload

    def historic_flight_positions_full(
        self,
        timestamp_utc: dt.datetime,
        flights: Optional[List[str]] = None,
        callsigns: Optional[List[str]] = None,
        registrations: Optional[List[str]] = None,
        airports: Optional[List[str]] = None,
        routes: Optional[List[str]] = None,
    ) -> List[Dict[str, Any]]:
        unix_ts = int(timestamp_utc.astimezone(dt.timezone.utc).timestamp())
    
        params = [("timestamp", str(unix_ts))]
    
        for value in flights or []:
            params.append(("flights", value))
        for value in callsigns or []:
            params.append(("callsigns", value))
        for value in registrations or []:
            params.append(("registrations", value))
        for value in airports or []:
            params.append(("airports", value))
        for value in routes or []:
            params.append(("routes", value))
    
        payload = self.get("/historic/flight-positions/full", params=params)
        return [x for x in ensure_list(payload) if isinstance(x, dict)]

In [55]:
def extract_flight_identity(item: Dict[str, Any]) -> Dict[str, Any]:
    fr24_id = item.get("fr24_id")
    if not fr24_id:
        raise RuntimeError("Could not extract FR24 flight identifier.")

    dep_time = parse_datetime_maybe(item.get("datetime_takeoff")) or parse_datetime_maybe(item.get("first_seen"))
    arr_time = parse_datetime_maybe(item.get("datetime_landed")) or parse_datetime_maybe(item.get("last_seen"))

    return {
        "fr24_id": fr24_id,
        "flight": item.get("flight"),
        "callsign": item.get("callsign"),
        "dep_iata": item.get("orig_iata"),
        "dep_icao": item.get("orig_icao"),
        "arr_iata": item.get("dest_iata_actual") or item.get("dest_iata"),
        "arr_icao": item.get("dest_icao_actual") or item.get("dest_icao"),
        "departure_time": dep_time,
        "arrival_time": arr_time,
        "first_seen": parse_datetime_maybe(item.get("first_seen")),
        "last_seen": parse_datetime_maybe(item.get("last_seen")),
        "raw": item,
    }

In [56]:
def local_departure_datetime(client: FR24Client, flight_item: Dict[str, Any]) -> Optional[dt.datetime]:
    dep_time_utc = parse_datetime_maybe(flight_item.get("datetime_takeoff")) or parse_datetime_maybe(flight_item.get("first_seen"))
    orig_iata = flight_item.get("orig_iata")

    if dep_time_utc is None or not orig_iata:
        return None

    tz_name = client.get_airport_timezone(orig_iata)
    if not tz_name:
        return None

    try:
        return dep_time_utc.astimezone(ZoneInfo(tz_name))
    except Exception:
        return None


def choose_best_flight_by_local_departure_date(
    client: FR24Client,
    flights: List[Dict[str, Any]],
    requested_flight: str,
    requested_date: dt.date,
) -> Dict[str, Any]:
    exact_matches = []
    fallbacks = []

    for item in flights:
        if str(item.get("flight") or "").upper() != requested_flight.upper():
            continue

        dep_utc = parse_datetime_maybe(item.get("datetime_takeoff")) or parse_datetime_maybe(item.get("first_seen"))
        dep_local = local_departure_datetime(client, item)
        dep_local_date = dep_local.date() if dep_local else None

        row = {
            "item": item,
            "dep_utc": dep_utc,
            "dep_local_date": dep_local_date,
        }

        if dep_local_date == requested_date:
            exact_matches.append(row)
        else:
            fallbacks.append(row)

    if exact_matches:
        exact_matches.sort(key=lambda x: x["dep_utc"] or dt.datetime.max.replace(tzinfo=dt.timezone.utc))
        return exact_matches[0]["item"]

    def fallback_key(row):
        dep_local_date = row["dep_local_date"]
        if dep_local_date is None:
            return (999999, row["dep_utc"] or dt.datetime.max.replace(tzinfo=dt.timezone.utc))
        return (
            abs((dep_local_date - requested_date).days),
            row["dep_utc"] or dt.datetime.max.replace(tzinfo=dt.timezone.utc),
        )

    if fallbacks:
        fallbacks.sort(key=fallback_key)
        return fallbacks[0]["item"]

    raise RuntimeError("No suitable flight found.")

In [57]:
def preview_flight_candidates(client: FR24Client, flights: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = []

    for item in flights:
        dep_utc = parse_datetime_maybe(item.get("datetime_takeoff")) or parse_datetime_maybe(item.get("first_seen"))
        arr_utc = parse_datetime_maybe(item.get("datetime_landed")) or parse_datetime_maybe(item.get("last_seen"))
        dep_local = local_departure_datetime(client, item)
        tz_name = client.get_airport_timezone(item.get("orig_iata")) if item.get("orig_iata") else None

        rows.append({
            "fr24_id": item.get("fr24_id"),
            "flight": item.get("flight"),
            "callsign": item.get("callsign"),
            "orig_iata": item.get("orig_iata"),
            "dest_iata": item.get("dest_iata_actual") or item.get("dest_iata"),
            "dep_utc": dep_utc,
            "dep_local": dep_local,
            "dep_local_date": dep_local.date() if dep_local else None,
            "arr_utc": arr_utc,
            "timezone": tz_name,
        })

    rows.sort(key=lambda x: x["dep_utc"] or dt.datetime.max.replace(tzinfo=dt.timezone.utc))
    return rows

In [58]:
def find_flight_for_date(
    client: FR24Client,
    flight_number: str,
    date_str: str,
    search_padding_hours: int = 18,
    limit: int = 50,
) -> Dict[str, Any]:
    requested_date = dt.date.fromisoformat(date_str)

    # Broad UTC search window. Final selection is done by local departure date.
    day_start_utc = dt.datetime.combine(requested_date, dt.time.min, tzinfo=dt.timezone.utc)
    day_end_utc = day_start_utc + dt.timedelta(days=1)

    search_start = day_start_utc - dt.timedelta(hours=search_padding_hours)
    search_end = day_end_utc + dt.timedelta(hours=search_padding_hours)

    summaries = client.flight_summary_full(
        flight_number=flight_number,
        start_utc=search_start,
        end_utc=search_end,
        limit=limit,
    )

    chosen = choose_best_flight_by_local_departure_date(
        client=client,
        flights=summaries,
        requested_flight=flight_number,
        requested_date=requested_date,
    )

    return extract_flight_identity(chosen)

In [59]:
def extract_track_list(track_raw: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not isinstance(track_raw, list) or not track_raw:
        raise RuntimeError("flight-tracks response is empty or not a list")

    first = track_raw[0]
    tracks = first.get("tracks")

    if not isinstance(tracks, list):
        raise RuntimeError("Could not find 'tracks' in flight-tracks response")

    return tracks


def parse_track_points(track_raw: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    raw_points = extract_track_list(track_raw)
    points = []

    for item in raw_points:
        ts = parse_datetime_maybe(item.get("timestamp"))
        lat = item.get("lat")
        lon = item.get("lon")
        alt = item.get("alt")

        if ts is None or lat is None or lon is None:
            continue

        try:
            lat = float(lat)
            lon = float(lon)
        except Exception:
            continue

        try:
            alt = float(alt) if alt is not None else None
        except Exception:
            alt = None

        points.append({
            "timestamp": ts,
            "lat": lat,
            "lon": lon,
            "alt": alt,
            "gspeed": item.get("gspeed"),
            "vspeed": item.get("vspeed"),
            "track": item.get("track"),
            "callsign": item.get("callsign"),
            "source": item.get("source"),
        })

    points.sort(key=lambda x: x["timestamp"])
    return points

In [60]:
def resample_points(points: List[Dict[str, Any]], interval_seconds: int = 60) -> List[Dict[str, Any]]:
    if not points:
        return []

    sampled = [points[0]]
    last_ts = points[0]["timestamp"]

    for pt in points[1:]:
        ts = pt["timestamp"]
        if (ts - last_ts).total_seconds() >= interval_seconds:
            sampled.append(pt)
            last_ts = ts

    if sampled[-1] != points[-1]:
        sampled.append(points[-1])

    return sampled

In [61]:
def write_gpx(
    points: List[Dict[str, Any]],
    output_file: str,
    track_name: str,
    description: str,
) -> None:
    if not points:
        raise RuntimeError("No valid points to write.")

    gpx = ET.Element(
        "gpx",
        attrib={
            "version": "1.1",
            "creator": "fr24-notebook-export",
            "xmlns": "http://www.topografix.com/GPX/1/1",
        },
    )

    metadata = ET.SubElement(gpx, "metadata")
    ET.SubElement(metadata, "name").text = track_name
    ET.SubElement(metadata, "desc").text = description
    ET.SubElement(metadata, "time").text = iso_z(points[0]["timestamp"])

    trk = ET.SubElement(gpx, "trk")
    ET.SubElement(trk, "name").text = track_name
    ET.SubElement(trk, "desc").text = description

    trkseg = ET.SubElement(trk, "trkseg")

    for pt in points:
        trkpt = ET.SubElement(
            trkseg,
            "trkpt",
            attrib={
                "lat": f'{pt["lat"]:.8f}',
                "lon": f'{pt["lon"]:.8f}',
            },
        )
        if pt["alt"] is not None:
            ET.SubElement(trkpt, "ele").text = f'{pt["alt"]:.2f}'
        ET.SubElement(trkpt, "time").text = iso_z(pt["timestamp"])

    tree = ET.ElementTree(gpx)
    ET.indent(tree, space="  ", level=0)
    tree.write(output_file, encoding="utf-8", xml_declaration=True)

In [160]:
def export_flight_to_gpx(
    client: FR24Client,
    flight_number: str,
    date_str: str,
    output_file: str,
    interval_seconds: int = 60,
) -> Dict[str, Any]:
    flight_info = find_flight_for_date(
        client=client,
        flight_number=flight_number,
        date_str=date_str,
    )

    track_raw = client.flight_tracks(flight_info["fr24_id"])
    points_all = parse_track_points(track_raw)
    points = points_all

    description = (
        f'{flight_info["flight"]} '
        f'{flight_info["dep_iata"]}->{flight_info["arr_iata"]} | '
        f'dep={iso_z(flight_info["departure_time"])} '
        f'arr={iso_z(flight_info["arrival_time"])} | '
        f'fr24_id={flight_info["fr24_id"]}'
    )

    write_gpx(
        points=points,
        output_file=output_file,
        track_name=str(flight_info["flight"] or flight_number),
        description=description,
    )

    return {
        "flight_info": flight_info,
        "track_raw": track_raw,
        "points_all": points_all,
        "points": points,
        "output_file": output_file,
    }

In [175]:
def scan_historic_day_for_flight(
    client: FR24Client,
    flight_number: str,
    date_str: str,
    step_minutes: int = 5,
    airports: Optional[List[str]] = None,
    routes: Optional[List[str]] = None,
) -> List[Dict[str, Any]]:
    day = dt.date.fromisoformat(date_str)
    start_utc = dt.datetime.combine(day, dt.time.min, tzinfo=dt.timezone.utc)
    end_utc = start_utc + dt.timedelta(days=1)

    hits = []
    t = start_utc

    while t < end_utc:
        rows = client.historic_flight_positions_full(
            timestamp_utc=t,
            flights=[flight_number],
            airports=airports,
            routes=routes,
        )
        hits.extend(rows)
        t += dt.timedelta(minutes=step_minutes)

    return hits

In [180]:
def parse_points_from_historic_snapshots(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    points = []
    seen = set()

    for item in rows:
        ts = parse_datetime_maybe(
            item.get("timestamp")
            or item.get("datetime")
            or item.get("time")
        )
        lat = item.get("lat") if "lat" in item else item.get("latitude")
        lon = item.get("lon") if "lon" in item else item.get("longitude")
        alt = item.get("alt") if "alt" in item else item.get("altitude")

        if ts is None or lat is None or lon is None:
            continue

        try:
            lat = float(lat)
            lon = float(lon)
        except Exception:
            continue

        try:
            alt = float(alt) if alt is not None else None
        except Exception:
            alt = None

        key = (ts, lat, lon)
        if key in seen:
            continue
        seen.add(key)

        points.append({
            "timestamp": ts,
            "lat": lat,
            "lon": lon,
            "alt": alt,
            "flight": item.get("flight"),
            "callsign": item.get("callsign"),
            "registration": item.get("reg") or item.get("registration"),
            "fr24_id": item.get("fr24_id") or item.get("flight_id"),
        })

    points.sort(key=lambda x: x["timestamp"])
    return points

In [188]:
def export_track_to_gpx_historical(client, fr24_id, output_file, track_name=None, interval_seconds=None):
    track_raw = client.flight_tracks(fr24_id)
    points = parse_track_points(track_raw)

    if interval_seconds is not None:
        points = resample_points(points, interval_seconds=interval_seconds)

    if track_name is None:
        track_name = fr24_id

    write_gpx(
        points=points,
        output_file=output_file,
        track_name=track_name,
        description=f"FR24 flight-tracks export for {fr24_id}",
    )

    return points

In [197]:
def print_flight_info_by_fr24_id(client, fr24_id):
    track_raw = client.flight_tracks(fr24_id)

    if not isinstance(track_raw, list) or not track_raw:
        print("No data returned.")
        return

    flight = track_raw[0]
    tracks = flight.get("tracks", [])

    first_track = tracks[0] if tracks else {}
    last_track = tracks[-1] if tracks else {}

    print("FR24 ID:   ", flight.get("fr24_id"))
    print("Flight:    ", first_track.get("callsign") or "unknown")
    print("Points:    ", len(tracks))
    print("First time:", first_track.get("timestamp"))
    print("Last time: ", last_track.get("timestamp"))
    print("First pos: ", first_track.get("lat"), first_track.get("lon"))
    print("Last pos:  ", last_track.get("lat"), last_track.get("lon"))
    print("Source:    ", first_track.get("source"))

In [228]:
TOKEN = "019dabb6-3922-717a-b423-36cec415ae08|TZ6kAwLjoKWFcXqk8WHhnfYgMs91HevkogbvNK9N60554b83"
client = FR24Client(TOKEN, verbose=True)

In [200]:
result = export_flight_to_gpx(
    client=client,
    flight_number="YK-959",
    date_str="2022-04-05",
    output_file="flights/2022-04-05-moscow-yerevan.gpx",
    interval_seconds=1,
)

print(result["output_file"])
print("Flight:", result["flight_info"]["flight"])
print("Route:", result["flight_info"]["dep_iata"], "->", result["flight_info"]["arr_iata"])
print("Departure UTC:", result["flight_info"]["departure_time"])
print("Arrival UTC:", result["flight_info"]["arrival_time"])
print("Raw points:", len(result["points_all"]))
print("Resampled points:", len(result["points"]))

GET https://fr24api.flightradar24.com/api/flight-summary/full
params=[('flights', '3F322'), ('flight_datetime_from', '2022-04-04T06:00:00Z'), ('flight_datetime_to', '2022-04-06T18:00:00Z'), ('limit', '50')]
status = 400
{"message":"Validation failed","details":"Date range is not available. Please select a flight_datetime_from after 2022-06-01 00:00:00."}


HTTPError: 400 Client Error: Bad Request for url: https://fr24api.flightradar24.com/api/flight-summary/full?flights=3F322&flight_datetime_from=2022-04-04T06%3A00%3A00Z&flight_datetime_to=2022-04-06T18%3A00%3A00Z&limit=50

In [230]:
historic_rows = scan_historic_day_for_flight(
    client=client,
    flight_number="YK959",
    date_str="2022-03-14",
    step_minutes=30,
)

GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647216000'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647217800'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647219600'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647221400'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647223200'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-positions/full
params=[('timestamp', '1647225000'), ('flights', 'YK959')]
status = 200
{"data":[]}
GET https://fr24api.flightradar24.com/api/historic/flight-

KeyboardInterrupt: 

In [231]:
print_flight_info_by_fr24_id(client, "2b226da5")

GET https://fr24api.flightradar24.com/api/flight-tracks
params={'flight_id': '2b226da5'}
status = 200
[{"fr24_id":"2b226da5","tracks":[{"timestamp":"2022-03-14T14:12:44Z","lat":43.08255,"lon":74.47495,"alt":0,"gspeed":11,"vspeed":0,"track":261,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:12:50Z","lat":43.08255,"lon":74.4746,"alt":0,"gspeed":11,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:12:57Z","lat":43.08255,"lon":74.4741,"alt":0,"gspeed":13,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:06Z","lat":43.08255,"lon":74.47323,"alt":0,"gspeed":14,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:13Z","lat":43.08255,"lon":74.47272,"alt":0,"gspeed":15,"vspeed":0,"track":258,"squawk":"0205","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:20Z","lat":43.08255,"lon":74.47204,"al

In [232]:
points = export_track_to_gpx_historical(
    client=client,
    fr24_id="2b226da5",
    output_file="flights/2022-03-14-moscow-bishkek.gpx",
    track_name="YK959",
    interval_seconds=1,
)

GET https://fr24api.flightradar24.com/api/flight-tracks
params={'flight_id': '2b226da5'}
status = 200
[{"fr24_id":"2b226da5","tracks":[{"timestamp":"2022-03-14T14:12:44Z","lat":43.08255,"lon":74.47495,"alt":0,"gspeed":11,"vspeed":0,"track":261,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:12:50Z","lat":43.08255,"lon":74.4746,"alt":0,"gspeed":11,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:12:57Z","lat":43.08255,"lon":74.4741,"alt":0,"gspeed":13,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:06Z","lat":43.08255,"lon":74.47323,"alt":0,"gspeed":14,"vspeed":0,"track":258,"squawk":"0","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:13Z","lat":43.08255,"lon":74.47272,"alt":0,"gspeed":15,"vspeed":0,"track":258,"squawk":"0205","callsign":"AVJ959","source":"ADSB"},{"timestamp":"2022-03-14T14:13:20Z","lat":43.08255,"lon":74.47204,"al